<!-- WBPROJ_PATCHED -->
# Nigeria — *Big Brother Naija* S9, "Her Money Her Power" pipeline

End-to-end pipeline used to produce the canonical Nigeria datasets:

1. **Inspect** raw Nairaland scrape → `data/raw/nigeria/BBNaija_nairaland.csv` (21,155 rows)
2. **Clean & dedup**, preserving Tier hierarchy → `data/interim/nigeria/BBNaija_cleaned.xlsx`
3. **Keyword filter** with Nigerian context (English + Pidgin + Yoruba + Igbo + Hausa)
4. **Embedding-based rerank** for gender relevance → `data/interim/nigeria/BBNaija_gender_filtered_rerank.xlsx`
5. **Analysis-ready set** (922 rows, post-rerank Top-N) → `data/processed/nigeria/BBNaija_final_topic_relevant.xlsx`
6. **LLM coding** (themes + sentiment + emotion + sexism flag) on the 100-row focused subset → `data/processed/nigeria/BBNaija_for_llm_coding.xlsx`
7. **Validation** vs human coding → `data/human_coded/nigeria/BBNaija_human_coding_dataset.xlsx`

Path constants assume the notebook is run from `notebooks/`.
Install dependencies once: `pip install -e .` (from repo root).


In [ ]:
# Setup — load .env so OPENAI_API_KEY is available to OpenAI() automatically.
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set — see .env.example"

import json, re, time
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

client = OpenAI()  # reads OPENAI_API_KEY from env via .env


## 1. Raw Nairaland scrape

The Nairaland forum data preserves the parent/child reply tree via the `Tier`
column. Tier 0 = original post; Tier 1 = direct reply; etc. We retain this
context so downstream coding can interpret a comment relative to what it's
replying to.


In [ ]:
RAW = "../data/raw/nigeria/BBNaija_nairaland.csv"

df_raw = pd.read_csv(RAW)
print(f"raw shape: {df_raw.shape}")
print(f"columns: {list(df_raw.columns)}")
print()
print("tier distribution:")
print(df_raw["Tier"].value_counts().sort_index().head(10))
print()
print("sample (first 2):")
df_raw.head(2)


## 2. Clean & dedup

Cleaning steps mirror the report's Section 2.2 spec:
- lowercase, URL strip, whitespace normalize
- de-duplicate by `PostID`
- retain Pidgin/Yoruba/Igbo/Hausa code-switching, emojis, and slang (these encode tone)
- preserve `Tier` so reply context isn't lost


In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def normalize(text: str) -> str:
    if pd.isna(text): return ""
    s = str(text).lower()
    s = URL_RE.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df = df_raw.copy()
df["text"] = df["Comment"].fillna(df.get("Cleaned_Comment", "")).apply(normalize)

# de-dup by PostID (stable identifier on Nairaland)
before = len(df)
df = df.drop_duplicates(subset=["PostID"]).reset_index(drop=True)
print(f"de-dup: {before} -> {len(df)}")

# drop empty / very short posts (< 5 chars)
df = df[df["text"].str.len() >= 5].reset_index(drop=True)
print(f"after dropping empty: {len(df)}")

OUT_CLEAN = "../data/interim/nigeria/BBNaija_cleaned.xlsx"
df.to_excel(OUT_CLEAN, index=False)
print(f"saved: {OUT_CLEAN}")


## 3. Keyword filter (Nigerian context)

High-recall Boolean filter using the keyword families from the report's
Appendix A — Nigeria. We retain anything that hits at least one term.
The set is intentionally permissive: false positives get tightened in the
semantic rerank stage.


In [ ]:
# Subset of the report's Appendix A keyword bank — representative across sub-themes.
NIGERIAN_KEYWORDS = [
    # Local culture
    "tribe", "tradition", "marriage", "bride", "dowry", "stigma", "taboo", "religion", "church", "imam", "pastor",
    # Gender norms — general
    "gender", "patriarchy", "feminism", "feminist", "women", "girls", "men", "boys", "husband", "wife",
    "single mother", "breadwinner", "as a woman", "as a man",
    # Vulgarity (English + Pidgin)
    "bitch", "slut", "whore", "ashawo", "ashewo", "olosho", "runs girl", "mumu", "kpekus",
    # Femininity stereotypes
    "wife material", "slay queen", "pepper dem", "she sabi cook", "na good girl",
    # Masculinity stereotypes
    "real man", "man up", "alpha male", "big boy", "hustler", "odogwu",
    # Body / sexuality
    "flat chest", "yellow", "light skin", "ikebe", "yansh", "nyash", "orobo",
    # GBV
    "abuse", "rape", "harassment", "battered", "beat", "cheat", "domestic violence",
    # Empowerment
    "education", "career", "independent", "empowerment", "feminist movement",
    # Backlash / neosexism (Pidgin)
    "women wan dey rule", "men no get voice", "feminism don spoil",
    # BBNaija-specific
    "bbnaija", "biggie", "vote", "evict", "hmhp", "her money her power",
]

# Compile a single regex with word boundaries where applicable
escaped = [re.escape(k) for k in NIGERIAN_KEYWORDS]
KW_RE = re.compile(r"(?:" + "|".join(escaped) + ")", re.IGNORECASE)

def matched_terms(text: str) -> list[str]:
    return list(set(KW_RE.findall(text or "")))

df["matched_keywords"] = df["text"].apply(matched_terms)
df["kw_hit"] = df["matched_keywords"].apply(lambda lst: len(lst) > 0)

filtered = df[df["kw_hit"]].reset_index(drop=True)
print(f"keyword-hit candidates: {len(filtered)} of {len(df)}")
print(f"top 10 matched terms:")
all_hits = pd.Series([t for sub in filtered["matched_keywords"] for t in sub])
print(all_hits.value_counts().head(10))


## 4. Embedding-based semantic rerank

Per the report (§2.2 + Appendix C5), keyword filtering catches direct
mentions but misses implicit gender talk ("she shouldn't be acting like that"
without an explicit gender term). Semantic rerank with
`text-embedding-3-large` cosine similarity recovers those.

Embedding query is the concatenation of all keyword families — the centroid
captures the gender-discourse manifold.


In [ ]:
# Embedding model is gpt-5.1 family; embeddings are deterministic
# regardless of LLM temperature.
EMBED_MODEL = "text-embedding-3-large"

# Embed the keyword "prototype" query
query_text = " ".join(NIGERIAN_KEYWORDS)
query_emb = client.embeddings.create(model=EMBED_MODEL, input=query_text).data[0].embedding
query_emb = np.array(query_emb, dtype="float32").reshape(1, -1)

# Embed all keyword-hit candidates in batches
texts = filtered["text"].tolist()
batch_size = 128
emb_rows = []
for i in tqdm(range(0, len(texts), batch_size), desc="embedding"):
    batch = texts[i:i + batch_size]
    resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
    emb_rows.extend(r.embedding for r in resp.data)

emb = np.array(emb_rows, dtype="float32")
sims = cosine_similarity(emb, query_emb).flatten()

filtered["semantic_score"] = sims
filtered = filtered.sort_values("semantic_score", ascending=False).reset_index(drop=True)
filtered["semantic_rank"] = filtered.index + 1

OUT_RERANK = "../data/interim/nigeria/BBNaija_gender_filtered_rerank.xlsx"
filtered.to_excel(OUT_RERANK, index=False)
print(f"saved: {OUT_RERANK}  (top score: {filtered['semantic_score'].iloc[0]:.3f})")


## 5. Build the analysis-ready set

The report uses a fixed Top-N (922 retained for Nigeria) for reproducibility
rather than a calibrated threshold. We mirror that target.


In [ ]:
TOP_N = 922  # matches report's analysis-ready BBNaija sample size

ar = filtered.head(TOP_N).reset_index(drop=True)

# Keep only the columns we need downstream
keep = ["PostID", "Tier", "Username", "Timestamp", "text", "matched_keywords",
        "semantic_score", "semantic_rank"]
ar = ar[[c for c in keep if c in ar.columns]]

OUT_AR = "../data/processed/nigeria/BBNaija_analysis_ready.xlsx"
ar.to_excel(OUT_AR, index=False)
print(f"analysis-ready: {len(ar)} rows -> {OUT_AR}")
print()
print("score distribution:")
print(ar["semantic_score"].describe())


## 6. LLM coding — themes, sentiment, emotion (`gpt-5.1`)

Closed-set Nigerian theme taxonomy (9 themes, multi-label) from the report's
Appendix B.1, paired with 3-class sentiment and a single-label emotion tag.
Strict JSON output, validated.


In [ ]:
LLM_MODEL = "gpt-5.1"

NIGERIA_THEMES = [
    "Participation, Competition, and Influence",
    "Marriage, Relationships, and Loyalty Norms",
    "Sexuality and Respectability Policing",
    "Sexist or Derogatory Language",
    "Conflict, Insults, and Humiliation Dynamics",
    "Moral Judgment and Character Evaluation",
    "Gender Norms and Gender Double Standards",
    "Emotional Expression",
    "Behavioral Responses and Engagement",
]
SENTIMENTS = ["Positive", "Neutral", "Negative"]
EMOTIONS = ["Contempt", "Anger", "Disgust", "Sadness", "Shame",
            "Embarrassment", "Guilt", "Compassion", "Pride", "No emotion"]


def classify_comment(text: str, max_retries: int = 2) -> dict:
    prompt = f"""You are coding social-media comments about *Big Brother Naija* Season 9.

Comment:
\"\"\"{text}\"\"\"

Pick zero or more themes from this closed list (multi-label):
{NIGERIA_THEMES}

Pick exactly one sentiment from: {SENTIMENTS}
Pick exactly one dominant emotion from: {EMOTIONS}

Return ONLY valid JSON of the form:
{{"themes": [...], "sentiment": "...", "emotion": "..."}}"""

    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                temperature=0,
                messages=[{"role": "user", "content": prompt}],
            )
            content = resp.choices[0].message.content.strip()
            content = content.replace("```json", "").replace("```", "").strip()
            data = json.loads(content)
            # Schema validation
            assert isinstance(data.get("themes", []), list)
            assert data.get("sentiment") in SENTIMENTS
            assert data.get("emotion") in EMOTIONS
            return data
        except Exception:
            if attempt == max_retries:
                return {"themes": [], "sentiment": "Error", "emotion": "Error"}
            time.sleep(1.5)


# Run on the 100-row focused-coding subset (matches report Appendix H.3)
FOCUSED = "../data/processed/nigeria/BBNaija_for_llm_coding.xlsx"
focused = pd.read_excel(FOCUSED)
print(f"focused subset: {focused.shape}")

# Optionally re-run; or check if labels are already present
if "Themes" in focused.columns and focused["Themes"].notna().any():
    print("focused subset already has themes/sentiment labels — skipping LLM call.")
    coded = focused.copy()
else:
    print(f"running gpt-5.1 classification on {len(focused)} comments...")
    text_col = next((c for c in focused.columns if c.lower() == "comment text"), focused.columns[0])
    results = [classify_comment(t) for t in tqdm(focused[text_col].astype(str), desc="classify")]
    coded = focused.copy()
    coded["Themes"] = [", ".join(r.get("themes", [])) for r in results]
    coded["Sentiment"] = [r.get("sentiment", "") for r in results]
    coded["Emotion"] = [r.get("emotion", "") for r in results]

print(coded[["Themes", "Sentiment", "Emotion"]].head() if "Themes" in coded.columns else coded.head())


## 7. Sexism flag detection (`gpt-5.1`)

Yes/No flag plus a short evidence quote. Functionally similar to the India
sexism module documented in Appendix C7 ("optional modules").


In [ ]:
def detect_sexism(text: str) -> dict:
    prompt = f"""Does the following comment contain sexist or gender-derogatory language?

Comment:
\"\"\"{text}\"\"\"

Return ONLY valid JSON:
{{"sexist": "Yes" | "No", "quote": "<short verbatim quote, empty if no>"}}"""

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
        )
        content = resp.choices[0].message.content.strip()
        content = content.replace("```json", "").replace("```", "").strip()
        data = json.loads(content)
        if data.get("sexist") not in {"Yes", "No"}:
            data["sexist"] = "Unclear"
        return data
    except Exception:
        return {"sexist": "Error", "quote": ""}


# Run on the same focused 100-row subset
if "sexist_flag" not in coded.columns:
    print(f"running sexism detection on {len(coded)} comments...")
    text_col = next((c for c in coded.columns if c.lower() == "comment text"), coded.columns[0])
    sx = [detect_sexism(t) for t in tqdm(coded[text_col].astype(str), desc="sexism")]
    coded["sexist_flag"] = [r.get("sexist", "") for r in sx]
    coded["sexist_quote"] = [r.get("quote", "") for r in sx]

print(coded["sexist_flag"].value_counts() if "sexist_flag" in coded.columns else "(skipped)")


## 8. Language tagging (`gpt-4o-mini`)

Lightweight task — uses the cheap auxiliary model per `wbproj.config`.
Tags are *descriptive* (for sampling balance and error analysis), not used
as a hard exclusion.


In [ ]:
LIGHT_MODEL = "gpt-4o-mini"

LANG_LABELS = ["English", "Pidgin", "Yoruba", "Igbo", "Hausa", "Mixed"]


def tag_language(text: str) -> str:
    prompt = f"""Identify the dominant language of this Nigerian forum comment.
Pick exactly one of: {LANG_LABELS}.

Comment:
\"\"\"{text}\"\"\"

Return ONLY valid JSON: {{"language": "<one of the options>"}}"""

    try:
        resp = client.chat.completions.create(
            model=LIGHT_MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": "You are a precise language classifier for short social media comments."},
                {"role": "user", "content": prompt},
            ],
        )
        content = resp.choices[0].message.content.strip()
        content = content.replace("```json", "").replace("```", "").strip()
        data = json.loads(content) if content.startswith("{") else {"language": content}
        lang = data.get("language", "Unknown")
        return lang if lang in LANG_LABELS else "Mixed"
    except Exception:
        return "Unknown"


# Tag a small batch as a demonstration
sample = coded.head(20).copy()
text_col = next((c for c in sample.columns if c.lower() == "comment text"), sample.columns[0])
sample["language"] = [tag_language(t) for t in tqdm(sample[text_col].astype(str), desc="lang")]
sample["language"].value_counts()


## 9. Validate LLM coding against human gold

Compute the same metrics the report uses (Section 3.4): Accuracy, Precision,
Recall, F1, plus Cohen's κ. The 100-row LLM and human-coded subsets share the
same `Comment ID`s, so we align on that key.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score

HUMAN = "../data/human_coded/nigeria/BBNaija_human_coding_dataset.xlsx"
human = pd.read_excel(HUMAN)
print(f"human-coded shape: {human.shape}")

# Align on Comment ID
human_id = next((c for c in human.columns if c.strip().lower() == "comment id"), None)
llm_id = next((c for c in coded.columns if c.strip().lower() == "comment id"), None)
print(f"join key: human='{human_id}' llm='{llm_id}'")

if human_id and llm_id:
    merged = coded.merge(human[[human_id, "Themes", "Sentiment"]].rename(
        columns={"Themes": "Themes_human", "Sentiment": "Sentiment_human"}
    ), left_on=llm_id, right_on=human_id, suffixes=("", "_h"))
    print(f"merged: {len(merged)} rows")

    # Sentiment agreement (3-class) — exact-match
    y_llm = merged["Sentiment"].astype(str).str.strip().str.lower()
    y_hum = merged["Sentiment_human"].astype(str).str.strip().str.lower()
    mask = y_llm.isin(["positive", "neutral", "negative"]) & y_hum.isin(["positive", "neutral", "negative"])

    if mask.sum() > 0:
        acc = accuracy_score(y_hum[mask], y_llm[mask])
        kappa = cohen_kappa_score(y_hum[mask], y_llm[mask])
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_hum[mask], y_llm[mask], average="macro", zero_division=0,
        )
        print()
        print(f"sentiment metrics (n={mask.sum()}):")
        print(f"  accuracy  = {acc:.3f}")
        print(f"  precision = {prec:.3f}")
        print(f"  recall    = {rec:.3f}")
        print(f"  f1        = {f1:.3f}")
        print(f"  cohen κ   = {kappa:.3f}")
    else:
        print("no overlapping sentiment labels to compare.")


## 10. Visualizations

The standard set the report uses in Appendix G — sentiment donut, top-themes
lollipop, theme × sentiment heatmap. Same templates the India and Kenya
notebooks emit.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

# Use the focused-coded subset which has Themes + Sentiment per row
plot_df = coded.copy()
text_col = next((c for c in plot_df.columns if c.lower() == "comment text"), plot_df.columns[0])

# Explode themes into one row per (comment, theme)
def to_list(val):
    if pd.isna(val): return []
    s = str(val).strip()
    try:
        parsed = json.loads(s) if s.startswith("[") else None
        if isinstance(parsed, list): return [str(t).strip() for t in parsed]
    except Exception:
        pass
    return [t.strip() for t in s.split(",") if t.strip()]

plot_df["theme_list"] = plot_df["Themes"].apply(to_list)
expanded = plot_df.explode("theme_list").rename(columns={"theme_list": "theme"})
expanded = expanded[expanded["theme"].notna() & (expanded["theme"] != "")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sentiment donut
sent_counts = plot_df["Sentiment"].value_counts()
axes[0].pie(sent_counts, labels=sent_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2"), wedgeprops=dict(width=0.4))
axes[0].set_title("Sentiment distribution — BBNaija (focused 100)")

# Top-10 themes (lollipop)
top_themes = expanded["theme"].value_counts().head(10).sort_values()
axes[1].hlines(y=range(len(top_themes)), xmin=0, xmax=top_themes.values, color="steelblue")
axes[1].plot(top_themes.values, range(len(top_themes)), "o", markersize=10, color="darkblue")
axes[1].set_yticks(range(len(top_themes)))
axes[1].set_yticklabels(top_themes.index)
axes[1].set_xlabel("Count")
axes[1].set_title("Top themes — BBNaija (focused 100)")
plt.tight_layout()
plt.show()

# Theme × sentiment heatmap
pivot = expanded.pivot_table(index="theme", columns="Sentiment", aggfunc="size", fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", cbar_kws={"label": "Count"})
plt.title("Theme × sentiment — BBNaija (focused 100)")
plt.tight_layout()
plt.show()
